# BeatBridge: Batch anhören

Dieses Notebook lädt einen Batch über sein `manifest.json`, zeigt die gerenderten Varianten und spielt sie direkt im Notebook ab. Es setzt keine zusätzlichen Widgets voraus.

Starten Sie Jupyter im Projekt-Root (`BeatBridge`) und führen Sie die Zellen von oben nach unten aus. Für einen anderen Batch nur `BATCH_DIR` in der nächsten Zelle ändern.

In [ ]:
from pathlib import Path
import json

from IPython.display import Audio, Markdown, display

# Für einen anderen Lauf diesen Ordner anpassen.
BATCH_DIR = Path("output/batches/help_to_breath_first")
MANIFEST_PATH = BATCH_DIR / "manifest.json"

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Manifest nicht gefunden: {MANIFEST_PATH.resolve()}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
variants = manifest["rendered_variants"]
candidates = {item["candidate_id"]: item for item in manifest["candidate_generation"]["candidates"]}

print(f"{len(candidates)} Kandidat(en), {len(variants)} gerenderte Variante(n)")
print(f"Renderer: {', '.join(manifest['renderers'])}")

In [ ]:
for index, variant in enumerate(variants):
    candidate = candidates[variant["candidate_id"]]
    print(
        f"[{index}] {variant['renderer']}: "
        f"{candidate['transition_length_bars']} Takte / "
        f"{candidate['transition_length_beats']} Beats — "
        f"{variant['output_file']}"
    )

In [ ]:
def play(index=0):
    """Spiele eine Variante über ihren Index aus der Liste oben ab."""
    variant = variants[index]
    candidate = candidates[variant["candidate_id"]]
    audio_path = BATCH_DIR / variant["output_file"]
    if not audio_path.is_file():
        raise FileNotFoundError(f"WAV-Datei nicht gefunden: {audio_path}")

    display(Markdown(
        f"### Variante {index}: `{variant['renderer']}`  \n"
        f"Kandidat `{variant['candidate_id']}`, "
        f"{candidate['transition_length_bars']} Takte / "
        f"{candidate['transition_length_beats']} Beats"
    ))
    return Audio(filename=str(audio_path))

play(0)

In [ ]:
def play_renderer(renderer):
    """Spiele die erste Variante eines Renderers ab, z. B. 'equal_power'."""
    for index, variant in enumerate(variants):
        if variant["renderer"] == renderer:
            return play(index)
    raise ValueError(f"Renderer nicht im Batch: {renderer}")

# Direkter A/B-Vergleich zum ersten Lauf:
play_renderer("equal_power")

## Hörnotizen

Beim Vergleich besonders auf Beat-Stabilität, wahrgenommene Lautheit im Übergang, klangliche Kontinuität, Energieverlauf und mögliche Vocal-Clashes achten. Auffälligkeiten mit Variantenindex, Kandidaten-ID und Renderer in `docs/Bemerkungen.md` festhalten.